# P05 – Screen-Time Personality
**Name:** HARSH JAGDISHBHAI BRAHMBHATT
**Roll No.:** D06
**Enrollment No.:** 24004501210038
**Division:** D

**Problem:** Predict a user's behaviour class (1–5) from app usage time, screen-on time, battery drain, number of apps, data usage, age, gender, OS and device model.
**Dataset:** https://www.kaggle.com/datasets/valakhorasani/mobile-device-usage-and-user-behavior-dataset
**Target (Y):** User Behavior Class (1–5)
**Model:** Support Vector Machine (SVM)
**Main metric:** Macro F1 (and how it changes after removing one feature)

## Step 1: Create your notebook and load the libraries
- Open Google Colab (or Jupyter) and create a new notebook.
- Rename it as RollNo_P05_Name.ipynb (example: H35_P05_Riya.ipynb).
- In the first cell write a heading (Markdown cell): project title, your name, roll no., enrollment no., division.
- Run the code cell below. If you see 'All libraries loaded!' you are ready.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, classification_report,
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, RocCurveDisplay)
from sklearn.svm import SVC
import warnings; warnings.filterwarnings("ignore")
print("All libraries loaded!")

✍️ **Write here:** Title of the project and your details.

**Project:** Screen-Time Personality  
**Name:** HARSH JAGDISHBHAI BRAHMBHATT  
**Roll No.:** D06  
**Enrollment No.:** 24004501210038  
**Division:** D

## Step 2: Load your dataset
- Make sure the CSV file is uploaded (Colab) or in the same folder (Jupyter).
- Run the code below. You should see the number of rows and columns, and the first 5 rows.

In [ ]:
from pathlib import Path
csv_path = Path("user_behavior_dataset.csv")
if not csv_path.exists():
    raise FileNotFoundError("Place user_behavior_dataset.csv in the same folder as this notebook, then run again.")
df = pd.read_csv(csv_path)
print(df.shape)      # (rows, columns)
df.head()

✍️ **Write here:** How many rows and columns does your dataset have? Where did it come from (Kaggle link)?

Dataset: Mobile Device Usage and User Behavior Dataset (Kaggle). The row and column counts are printed in Step 2 after loading the CSV.

## Step 3: Understand the data
- Look at every column: what does it mean, is it a number or text?
- Check missing values and the target classes.
- Make a DATA DICTIONARY table: Column name | Meaning | Type (number/text) | Keep or drop.

In [ ]:
df.info()                   # column names, types, non-null counts
df.describe()               # min, max, mean of number columns
print(df.isnull().sum())    # missing values in each column
print("Duplicates:", df.duplicated().sum())

✍️ **Write here:** Data dictionary table + 2 lines about what you noticed.

Data dictionary: Run `df.info()` to see the exact column names and data types. The input variables describe device usage, age, gender, operating system, and device model. Check Step 3 output for missing values and duplicates before interpreting their counts.

## Step 4: Prepare the target column (Y)
- Your target (what the model predicts) is: User Behavior Class (1–5).
- The target already exists in the data. Just check how many rows each class has.
- If one class is much bigger than others, the data is IMBALANCED. Write this down: it affects which metric you trust.

In [ ]:
TARGET = "User Behavior Class"
print(df[TARGET].value_counts())                  # how many rows in each class
print(df[TARGET].value_counts(normalize=True))    # same, as percentages

✍️ **Write here:** Classes of the target and their counts. Is it balanced or imbalanced?

The target is **User Behavior Class**, with labels 1–5. The actual class counts and balance are displayed by Step 4 when the CSV is loaded.

## Step 5: Clean the data
- Remove duplicate rows.
- Fill missing values: number columns with median, text columns with mode (most common value). Or drop a column if most of it is empty.
- Drop ID / name columns and any column that gives away the answer (leakage).
- Special note for YOUR dataset: Drop User ID. The class is closely linked to usage columns, so near-perfect scores are possible: explain in your report why, and test the models again after removing App Usage Time.

In [ ]:
df = df.drop_duplicates()
df = df.dropna(subset=[TARGET])

# Drop ID / name / leaking columns (write the real column names from your data)
df = df.drop(columns=["User ID"], errors="ignore")

for col in df.columns:
    if col == TARGET: continue
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].fillna(df[col].median())     # number column -> median
    else:
        df[col] = df[col].fillna(df[col].mode().iloc[0] if not df[col].mode().empty else "Unknown")    # text column -> most common value
df = df.dropna(subset=[TARGET])

print(df.isnull().sum().sum(), "missing values left")
print(df.shape)

✍️ **Write here:** What you removed or filled, and WHY (one line each).

Duplicate records were removed. Missing numeric values were filled using each column’s median; missing text values were filled using the mode (or `Unknown` if no mode exists). Rows with a missing target were removed, and User ID was dropped because it is an identifier.

## Step 6: Explore the data with charts (EDA)
- Run the charts below. Look at each chart carefully.
- Write 5 INSIGHTS in simple English. Example: 'Players with more gold at 10 minutes win more often.'

In [ ]:
sns.countplot(x=df[TARGET]); plt.title("Class balance"); plt.show()

num_cols = df.select_dtypes(include="number").columns
df[num_cols].hist(figsize=(14, 10), bins=30); plt.tight_layout(); plt.show()

# Box plot of one important numeric column by class (change the column name)
sns.boxplot(x=df[TARGET], y=df[num_cols[0]]); plt.show()

plt.figure(figsize=(12, 8))
sns.heatmap(df[num_cols].corr(), cmap="coolwarm"); plt.title("Correlation heat map"); plt.show()

✍️ **Write here:** 5 insights from the charts (one line each).

Inspect the charts after running the notebook with the CSV. Record five observed findings about class balance, numeric distributions, outliers, usage differences between classes, and correlations. Actual findings cannot be verified without the dataset.

## Step 7: Create a new feature, encode and select features
- Create at least ONE new column from existing columns (change the example line to real columns).
- One-hot encoding turns text columns into 0/1 columns.
- SelectKBest (filter method) keeps the most useful columns. Write down which columns were selected.

In [ ]:
# 1) NEW FEATURE: create at least one new column that makes sense for your data
#    example: df["new_feature"] = df["column_a"] / (df["column_b"] + 1)
#    (change the column names to real columns from YOUR dataset)
df["Usage per App"] = df["App Usage Time (min/day)"] / (df["Number of Apps Installed"] + 1)

# 2) Separate X (inputs) and y (target)
X = df.drop(columns=[TARGET])
le_y = LabelEncoder()
y = le_y.fit_transform(df[TARGET])
print("Classes:", list(le_y.classes_))     # class 0, class 1, ...

# 3) ENCODING: turn text columns into numbers (one-hot encoding)
X = pd.get_dummies(X, drop_first=True)
print(X.shape)

# 4) FEATURE SELECTION (filter method): keep the most useful columns
from sklearn.feature_selection import SelectKBest, f_classif
k = min(15, X.shape[1])
selector = SelectKBest(f_classif, k=k).fit(X, y)
scores = pd.Series(selector.scores_, index=X.columns).sort_values(ascending=False)
print(scores.head(20))
X = X[selector.get_feature_names_out()]
feature_names = list(X.columns)
print("Selected:", feature_names)

✍️ **Write here:** The new feature you created and why; the list of selected features.

New feature: **Usage per App** = App Usage Time (min/day) / (Number of Apps Installed + 1). This measures average app usage intensity while avoiding division by zero. The exact selected feature names are printed in Step 7.

## Step 8: Split into train and test + scale
- 80% of the data is used to TRAIN the model, 20% to TEST it.
- stratify=y keeps the same class ratio in train and test.
- Support Vector Machine (SVM) needs SCALING: all columns on the same scale. Fit the scaler ONLY on train data.

In [ ]:
# If your data has more than 20,000 rows, first take a sample (SVM/KNN are slow on big data):
# df = df.sample(20000, random_state=42)   <- do this BEFORE Step 7 and re-run Step 7

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.shape, X_test.shape)

# SCALING (must for Support Vector Machine (SVM)). Fit ONLY on train data, then apply to test.
X_train_raw, X_test_raw = X_train.copy(), X_test.copy()   # keep unscaled copy
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

✍️ **Write here:** Size of train and test sets.

The dataset is split into **80% training** and **20% testing**, stratified by target class. The exact row counts are printed in Step 8.

## Step 9: Train your model: Support Vector Machine (SVM)
- The evaluate() function below trains a model and calculates ALL metrics: accuracy, precision, recall, F1, specificity and AUC.
- Run it once for the BASELINE (default settings). You will see the confusion matrix and classification report.
- Use only this model in your project: Support Vector Machine (SVM).

In [ ]:
results = []     # every setting you try is stored here

def evaluate(name, model, show=False):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    spec_list = []
    for i in range(len(cm)):
        tn = cm.sum() - cm[i, :].sum() - cm[:, i].sum() + cm[i, i]
        fp = cm[:, i].sum() - cm[i, i]
        spec_list.append(tn / (tn + fp))
    spec = np.mean(spec_list)
    try:
        proba = model.predict_proba(X_test)
        auc = roc_auc_score(y_test, proba, multi_class="ovr", average="macro")
    except Exception:
        auc = None
    row = {"Setting": name,
           "Accuracy": accuracy_score(y_test, y_pred),
           "Precision": precision_score(y_test, y_pred, average="macro", zero_division=0),
           "Recall": recall_score(y_test, y_pred, average="macro", zero_division=0),
           "F1": f1_score(y_test, y_pred, average="macro", zero_division=0),
           "Specificity": spec, "AUC": auc}
    if show:
        ConfusionMatrixDisplay(cm, display_labels=le_y.classes_).plot(cmap="Blues")
        plt.title(name); plt.show()
        print(classification_report(y_test, y_pred, target_names=[str(c) for c in le_y.classes_]))

    print(row)
    return row

# BASELINE: your model with default settings
model = SVC(probability=True, random_state=42)
results.append(evaluate("Baseline", model, show=True))

✍️ **Write here:** Baseline results. Explain the confusion matrix in words (how many correct, how many wrong).

The baseline is an SVM classifier. Its accuracy, macro precision, macro recall, macro F1, specificity, AUC, and confusion matrix are printed by Step 9. Correct predictions lie on the confusion matrix diagonal; off-diagonal entries are errors.

## Step 10: Try different settings (tuning)
- Scale the features (SVM is sensitive to feature scale).
- Train SVC with kernel='linear' and kernel='rbf' and compare.
- Try C = 0.1, 1, 10 for the better kernel and record the scores.
- Use probability=True (or decision_function) to draw the ROC curve.
- Every setting is added to the results list automatically.

In [ ]:
# A) Kernel: linear vs rbf
for kern in ["linear", "rbf"]:
    results.append(evaluate(f"kernel={kern}", SVC(kernel=kern, probability=True, random_state=42)))

# B) C value for the better kernel (change "rbf" if linear was better)
for c in [0.1, 1, 10]:
    results.append(evaluate(f"rbf, C={c}", SVC(kernel="rbf", C=c, probability=True, random_state=42), show=(c == 1)))

✍️ **Write here:** What changed when you changed each setting? Which one was best?

Step 10 compares linear and RBF kernels and RBF C values 0.1, 1, and 10. Compare the displayed macro F1 scores to identify the best configuration; the winner cannot be determined without running the dataset.

## Step 11: Compare all settings and pick your final model
- The table shows every setting you tried, side by side.
- Your most important metric is Macro F1 (and how it changes after removing one feature). Choose the setting with the best F1.
- Write your best setting inside final_model = ...( ) and run it.

In [ ]:
res = pd.DataFrame(results).round(3)
display(res)                                    # the comparison table for your report

res.set_index("Setting")[["Accuracy", "Precision", "Recall", "F1"]].plot(kind="bar", figsize=(10, 4))
plt.title("Comparison of settings"); plt.ylim(0, 1); plt.xticks(rotation=45, ha="right"); plt.show()

best = res.sort_values("F1", ascending=False).iloc[0]
print("Best setting by F1:", best["Setting"])

# FINAL MODEL: write the best setting here (example below), then show its full result
best_setting = best["Setting"]
if best_setting.startswith("kernel="):
    final_model = SVC(kernel=best_setting.split("=")[1], probability=True, random_state=42)
elif best_setting.startswith("rbf, C="):
    final_model = SVC(kernel="rbf", C=float(best_setting.split("C=")[1]), probability=True, random_state=42)
else:
    final_model = SVC(probability=True, random_state=42)
evaluate("FINAL", final_model, show=True)

✍️ **Write here:** Comparison table + bar chart + 3 lines: why you chose this final setting (talk about Macro F1 (and how it changes after removing one feature)).

The best configuration is selected by the highest measured macro F1 in Step 11, then fitted as the final SVM model. The comparison chart and exact performance scores appear after execution.

## Step 12: Predict 3 new inputs
- Make 3 new, realistic examples and let your final model predict them.
- Check: does the prediction make sense? Explain in one line each.

In [ ]:
# Start from 3 real rows and change some values to make 3 NEW realistic inputs
new = X.iloc[[0, 1, 2]].copy()

# Convert numeric columns to float before assigning decimal values.
# This prevents pandas LossySetitemError / TypeError for int64 columns.
numeric_features = new.select_dtypes(include="number").columns
new[numeric_features] = new[numeric_features].astype("float64")

# Modify one selected numeric feature in each example while keeping values realistic.
for i in range(3):
    if len(numeric_features):
        col = numeric_features[i % len(numeric_features)]
        row_index = new.index[i]
        new.loc[row_index, col] = float(new.loc[row_index, col]) * (1 + 0.05 * (i + 1))

display(new)
pred = final_model.predict(scaler.transform(new))
print("Predictions:", le_y.inverse_transform(pred))

✍️ **Write here:** The 3 inputs, the predictions, and whether they make sense.

Step 12 produces three predictions from modified dataset examples. Read the displayed inputs and predicted classes before deciding whether each prediction makes sense.

## Step 13: Write the report and prepare for viva
- Run the whole notebook from top to bottom: Runtime → Restart and run all (Colab) or Kernel → Restart & Run All (Jupyter). Fix any error.
- Make a 4–6 page PDF report (structure is in 'What to submit').
- Prepare the viva questions given below. You must be able to explain EVERY line of your code.

✍️ **Write here:** Conclusion: what your model can do, how good it is, and what you learned.

This project trains a Support Vector Machine to predict User Behavior Class from device usage features. Performance is assessed using macro F1 and other classification metrics. The final accuracy and conclusions require running the notebook with the CSV.